# Oxta — Validação FULL2 CLEAN (CHRASS40 + TTT + VIB, sem K·m)

**Objetivo:** validar a hipótese de que **CHRASS40 + TTT + Pantheon VIB** combinados
somam (ou superam) o ganho do melhor individual (CHRASS40 = −18.5% vs baseline).

**Por que sem K·m?** No cascade anterior, K·m+MoE foi o ÚNICO individual que machucou
(+5.8%) — provavelmente expert collapse em 10M params com 4 experts. O FULL anterior
(`full_d40` com K·m) ficou em +7.0% porque K·m arrastou. Esse novo FULL2 testa sem ele.

**Hipóteses:**
- 🟢 Se `full2_clean` < 3.3 (melhor que chrass40 alone): efeito ADITIVO confirmado
- 🟡 Se 3.3 ≤ `full2_clean` < 3.8: cancelamento parcial, ainda melhor que baseline
- 🔴 Se `full2_clean` ≥ 3.8: techs conflitam — usar só CHRASS40 no bundle do amigo

**3 runs apenas** (~45 min em T4 grátis):
1. `baseline` — referência
2. `chrass40` — campeão individual
3. `full2_clean` — CHRASS40 + TTT + VIB combinados (sem MoE)

**Modelo:** 6L × d=256 × 4H × vocab=8192 ≈ 10M params, 1500 steps.

In [ ]:
!nvidia-smi | head -10
!pip install --quiet pybind11 numpy zstandard matplotlib

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_OWNER = 'vitorGgC569'
REPO_NAME = 'reimagined'
BRANCH = 'oxta-contabil'
REPO_ROOT = Path('/content/reimagined-main')

clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        clone_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
        print('[auth] GITHUB_TOKEN OK')
except Exception:
    pass

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                    clone_url, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'remote', 'set-url',
                    'origin', clone_url], check=False)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard',
                    f'origin/{BRANCH}'], check=True)

os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

In [ ]:
# Build CUDA-enabled nsos_ext (T4/sm_75) — ~10 min na primeira execução
BUILD_DIR = REPO_ROOT / 'OXN/nsos/build-colab'
BUILD_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['cmake',
    '-S', str(REPO_ROOT / 'OXN/nsos'),
    '-B', str(BUILD_DIR),
    '-DNSOS_ENABLE_CUDA=ON',
    '-DNSOS_BUILD_PYTHON=ON',
    '-DNSOS_BUILD_TESTS=OFF',
    '-DNSOS_BUILD_CLI=OFF',
    '-DNSOS_BUILD_API=OFF',
    '-DNSOS_BUILD_OXTAMEM=OFF',
    '-DNSOS_CUDA_ARCHITECTURES=75',
], check=True)
subprocess.run(['cmake', '--build', str(BUILD_DIR),
    '--target', 'nsos_ext', '-j'], check=True)
print('Build OK')

In [ ]:
# Knobs + helper
LAYERS, D_MODEL, N_HEADS, VOCAB = 6, 256, 4, 8192
STEPS = 1500
SEED = 42
OUT_DIR = '/content/full2_reports'

!mkdir -p {OUT_DIR}
PROBE = str(REPO_ROOT / 'OXN/nsos/scripts/probe_cascade.py')
ANALYZE = str(REPO_ROOT / 'OXN/nsos/scripts/analyze_cascade.py')
PYTHONPATH_ENV = str(BUILD_DIR)

import time
def run_probe(label, *args):
    out_path = Path(OUT_DIR) / f'probe_{label}_s{SEED}.json'
    if out_path.exists():
        print(f'\n[SKIP] {label} (já existe em {out_path})')
        return
    print(f'\n{"="*60}\n  RUN: {label}\n{"="*60}')
    t0 = time.time()
    cmd = ['python', PROBE, '--label', label, '--seed', str(SEED),
           '--steps', str(STEPS), '--layers', str(LAYERS),
           '--d-model', str(D_MODEL), '--n-heads', str(N_HEADS),
           '--vocab-size', str(VOCAB),
           '--out-dir', OUT_DIR, '--cuda'] + list(args)
    env = {**os.environ, 'PYTHONPATH': PYTHONPATH_ENV}
    p = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(p.stdout[-2000:])
    if p.returncode != 0:
        print('STDERR:', p.stderr[-1000:])
    print(f'  done in {(time.time()-t0)/60:.1f} min')

## Run 1 — Baseline (referência)

In [ ]:
run_probe('baseline')

## Run 2 — CHRASS density=40% (campeão individual)

In [ ]:
run_probe('chrass40', '--use-chrass', '--chrass-density', '40')

## Run 3 — FULL2 CLEAN (CHRASS40 + TTT + VIB, **sem** K·m/MoE)

Esse é o teste da hipótese: 3 techs validadas individualmente combinadas.

In [ ]:
run_probe('full2_clean',
          '--use-chrass', '--chrass-density', '40',
          '--use-ttt',
          '--vib-beta', '0.01')

## Análise + plot

In [ ]:
subprocess.run(['python', ANALYZE,
    '--reports-dir', OUT_DIR,
    '--plot'], check=True)

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUT_DIR}/cascade_curves.png'))

In [ ]:
# Verdict final: full2_clean vs baseline vs chrass40
import json, statistics, glob

reports = {}
for f in glob.glob(f'{OUT_DIR}/probe_*.json'):
    r = json.loads(open(f).read())
    reports[r['label']] = r

base = reports.get('baseline')
c40 = reports.get('chrass40')
full2 = reports.get('full2_clean')

if base and full2:
    bf = statistics.fmean(base['loss']['all'][-200:])
    ff = statistics.fmean(full2['loss']['all'][-200:])
    delta_full2_vs_base = (ff - bf) / abs(bf) * 100
    print(f'\nbaseline      final: {bf:.4f}')
    print(f'full2_clean   final: {ff:.4f}   delta vs base: {delta_full2_vs_base:+.1f}%')
    if c40:
        cf = statistics.fmean(c40['loss']['all'][-200:])
        delta_c40 = (cf - bf) / abs(bf) * 100
        delta_full2_vs_c40 = (ff - cf) / abs(cf) * 100
        print(f'chrass40      final: {cf:.4f}   delta vs base: {delta_c40:+.1f}%')
        print(f'\n>>> full2 vs chrass40 (champion):  {delta_full2_vs_c40:+.1f}%')
        print()
        if delta_full2_vs_c40 < -2:
            print('  *** HIPÓTESE CONFIRMADA: combinação SOMA acima do campeão individual')
        elif delta_full2_vs_c40 < 1:
            print('  =   Empate técnico — combinação não atrapalha mas também não soma muito')
        else:
            print('  !!! Combinação ATRAPALHA o CHRASS40 alone — usar só CHRASS40 no bundle')
        print()
        if delta_full2_vs_base < -15:
            print(f'  >>> RECOMENDACAO BUNDLE: CHRASS40 + TTT + VIB (ganho {-delta_full2_vs_base:.1f}%)')
        elif delta_c40 < -15:
            print(f'  >>> RECOMENDACAO BUNDLE: SO CHRASS40 (ganho {-delta_c40:.1f}%)')
        else:
            print('  >>> RECOMENDACAO: revisar — nem combinacao nem chrass40 entregaram >15% melhoria')

In [ ]:
# Download
!cd /content && zip -r full2_reports.zip full2_reports/
from google.colab import files
files.download('/content/full2_reports.zip')